# Implementasi Minggu 12

Notebook ini memuat pembacaan data, pengolahan citra, dan penentuan status petak pada tahap implementasi awal. Output teks menggunakan data sintetis. Grafik dapat dibuat ulang dengan menjalankan seluruh sel. Hasil data PKLot asli tersedia pada notebook evaluasi 25 November di halaman utama repositori.


# Fiveclock · Implementasi Minggu 12

Deteksi status **kosong** dan **terisi mobil** dari citra statis kamera tetap. Notebook menampilkan baseline selisih citra dan jalur lengkap dengan normalisasi P5–P95, selisih terhadap acuan kosong, Canny, OR, dan morfologi. Output yang tersimpan berasal dari **demo sintetis**.

### Jalankan di Colab

1. Unggah notebook ke Colab dan jalankan semua sel dengan `MODE = "demo"` untuk melihat alurnya.
2. Untuk foto asli, ekstrak [PKLot UFPR04](https://web.inf.ufpr.br/vri/databases/parking-lot-database/) ke Google Drive. Pilih satu foto uji dan pasangannya (JPG/XML), lalu pilih satu atau beberapa foto acuan kosong beserta XML. Pastikan semua foto berasal dari kamera tetap yang sama.
3. Atur `MODE = "pklot"`, `DATA_ROOT`, `TEST_IMAGE`, dan `EMPTY_REFERENCES`. Isi ID petak yang dinilai. Jika satu foto acuan memuat semua petak kosong, gunakan path yang sama untuk setiap ID; jika perlu, gunakan acuan berbeda per petak. Cek secara visual bahwa petak uji hanya kosong atau berisi mobil. Lewati petak berisi motor, kendaraan lain, atau yang meragukan.
4. Jika dependensi belum tersedia, jalankan `%pip install numpy scipy pillow matplotlib` pada sel sementara.

Ambang pada notebook ini dipakai untuk **uji implementasi**. Untuk eksperimen Minggu 13, pilih ambang menggunakan data pengembangan dan kunci sebelum mengolah data uji. Notebook ini menguji alur klasik, sedangkan notebook evaluasi memuat pembanding HOG + SVM dan ablasi.

In [1]:
from pathlib import Path
import csv
import math
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

MODE = "demo"
DATA_ROOT = Path("/content/drive/MyDrive/UFPR04")
TEST_IMAGE = "Sunny/tanggal/foto_uji.jpg"
EMPTY_REFERENCES = {
    "1": "Sunny/tanggal/foto_acuan.jpg",
    "2": "Sunny/tanggal/foto_acuan.jpg",
}
WORK_SIZE = (960, 540)
DIFF_THRESHOLD = 25
OCCUPANCY_THRESHOLD = 0.20
SEED = 42
SOURCE_LABEL = {"demo": "DEMO SINTETIS", "pklot": "PKLot UFPR04"}
if MODE not in SOURCE_LABEL:
    raise ValueError('MODE harus "demo" atau "pklot".')

In [2]:
def generate_demo(root):
    """Buat fixture JPG dan XML untuk uji alur, dengan cahaya dan bayangan bervariasi."""
    rng = np.random.default_rng(SEED)
    for day in range(1, 2):
        weather = ("Cloudy", "Sunny", "Rainy")[(day - 1) % 3]
        folder = root / weather / f"2012-12-{day:02d}"
        folder.mkdir(parents=True, exist_ok=True)
        for index in range(3):
            shade = rng.integers(-22, 25)
            pixels = np.clip(80 + shade + rng.normal(0, 4, (240, 420)), 0, 255).astype(np.uint8)
            image = Image.fromarray(pixels)
            draw = ImageDraw.Draw(image)
            annotation = ET.Element("parking")
            for slot in range(4):
                x = slot * 100 + 20
                occupied = int((index + slot) % 3 == 0)
                draw.rectangle((x, 45, x + 82, 154), outline=135, width=2)
                if (index + slot) % 4 == 0:
                    draw.rectangle((x + 1, 90, x + 81, 153), fill=40 + int(shade / 2))
                if occupied:
                    offset = int(rng.integers(-5, 6))
                    body = int(rng.choice([90, 122, 172, 200]))
                    draw.rounded_rectangle((x + 15 + offset, 61, x + 65 + offset, 141),
                                           radius=8, fill=body, outline=55, width=2)
                    draw.rectangle((x + 20 + offset, 77, x + 60 + offset, 91), fill=48)
                    draw.rectangle((x + 20 + offset, 115, x + 60 + offset, 125), fill=57)
                space = ET.SubElement(annotation, "space", id=str(slot + 1), occupied=str(occupied))
                contour = ET.SubElement(space, "contour")
                for px, py in ((x, 45), (x + 82, 45), (x + 82, 154), (x, 154)):
                    ET.SubElement(contour, "point", x=str(px), y=str(py))
            stem = folder / f"2012-12-{day:02d}_10_{index:02d}_00"
            image.save(stem.with_suffix(".jpg"))
            ET.ElementTree(annotation).write(stem.with_suffix(".xml"))



if MODE == "demo":
    DATA_ROOT = Path("fiveclock_demo_m12")
    generate_demo(DATA_ROOT)
    demo_files = sorted(DATA_ROOT.rglob("*.jpg"))
    relative = [path.relative_to(DATA_ROOT).as_posix() for path in demo_files]
    TEST_IMAGE = relative[0]
    EMPTY_REFERENCES = {"1": relative[1], "2": relative[1],
                        "3": relative[2], "4": relative[1]}
else:
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount("/content/drive")

if not EMPTY_REFERENCES:
    raise ValueError("Isi minimal satu ID petak dan path foto acuannya.")

## 1. Baca XML dan petakan posisi petak

Koordinat tetap berasal dari XML acuan. Label XML foto uji ditampilkan sebagai pembanding setelah prediksi.

In [3]:
def parse_slots(image_path):
    xml_path = image_path.with_suffix(".xml")
    if not image_path.is_file() or not xml_path.is_file():
        raise FileNotFoundError(f"Butuh pasangan JPG/XML: {image_path}")
    slots = {}
    for item in ET.parse(xml_path).getroot().findall(".//space"):
        slot = item.get("id")
        if slot is not None:
            slots[slot] = item
    return slots


def polygon_points(slot):
    points = [(float(p.attrib["x"]), float(p.attrib["y"]))
              for p in slot.findall("contour/point")]
    if len(points) >= 3:
        return points
    center = slot.find("rotatedRect/center")
    size = slot.find("rotatedRect/size")
    angle = slot.find("rotatedRect/angle")
    if any(x is None for x in (center, size, angle)):
        raise ValueError(f"Koordinat petak {slot.get('id')} tidak lengkap.")
    cx, cy = float(center.attrib["x"]), float(center.attrib["y"])
    w, h = float(size.attrib["w"]) / 2, float(size.attrib["h"]) / 2
    a = math.radians(float(angle.attrib["d"]))
    return [(cx + x * math.cos(a) - y * math.sin(a),
             cy + x * math.sin(a) + y * math.cos(a))
            for x, y in ((-w, -h), (w, -h), (w, h), (-w, h))]


def read_image(path):
    with Image.open(path) as image:
        return image.convert("RGB").resize(WORK_SIZE, Image.Resampling.BILINEAR), image.size


test_path = DATA_ROOT / TEST_IMAGE
test_slots = parse_slots(test_path)
test_rgb, test_size = read_image(test_path)
test_gray = np.asarray(test_rgb.convert("L"), dtype=np.float32)
references, polygons, masks = {}, {}, {}
for slot, relative_path in EMPTY_REFERENCES.items():
    reference_path = DATA_ROOT / relative_path
    if reference_path.resolve() == test_path.resolve():
        raise ValueError(f"Petak {slot}: gunakan foto acuan yang berbeda dari foto uji.")
    ref_annotation = parse_slots(reference_path).get(slot)
    if ref_annotation is None or ref_annotation.get("occupied") != "0":
        raise ValueError(f"Petak {slot} harus berlabel kosong di XML acuan: {reference_path.name}")
    ref_rgb, (width, height) = read_image(reference_path)
    if (width, height) != test_size:
        raise ValueError(f"Ukuran foto uji dan acuan berbeda pada petak {slot}.")
    points = [(round(x * WORK_SIZE[0] / width), round(y * WORK_SIZE[1] / height))
              for x, y in polygon_points(ref_annotation)]
    canvas = Image.new("1", WORK_SIZE)
    ImageDraw.Draw(canvas).polygon(points, fill=1)
    mask = np.asarray(canvas, dtype=bool)
    if not mask.any():
        raise ValueError(f"Poligon petak {slot} berada di luar citra.")
    masks[slot], polygons[slot] = mask, points
    references[slot] = np.asarray(ref_rgb.convert("L"), dtype=np.float32)

## 2. Jalankan baseline dan metode klasik lengkap

Kedua jalur memakai ambang awal yang sama supaya keluaran tiap tahap mudah diperiksa. Canny memakai gradien Sobel, penipisan tepi dan histeresis melalui SciPy.

In [4]:
PIXEL_KERNEL = np.ones((3, 3), dtype=bool)


def normalize(gray):
    low, high = np.percentile(gray[normalization_mask], (5, 95))
    if high <= low:
        return gray
    return np.clip((gray - low) * (255 / (high - low)), 0, 255)


def detect_edges(gray):
    """Canny with Sobel gradients, directional suppression, and hysteresis."""
    smooth = ndi.gaussian_filter(gray.astype(np.float32), 1.0)
    gx, gy = ndi.sobel(smooth, axis=1), ndi.sobel(smooth, axis=0)
    magnitude = np.hypot(gx, gy)
    direction = np.round(np.arctan2(gy, gx) * (4 / np.pi)).astype(int) % 4
    keep = np.zeros_like(gray, dtype=bool)
    for angle, (dy, dx) in enumerate(((0, 1), (1, 1), (1, 0), (1, -1))):
        before = np.roll(magnitude, (dy, dx), (0, 1))
        after = np.roll(magnitude, (-dy, -dx), (0, 1))
        keep |= (direction == angle) & (magnitude >= before) & (magnitude >= after)
    keep[[0, -1], :] = False
    keep[:, [0, -1]] = False
    return ndi.binary_propagation(keep & (magnitude >= 150),
                                  structure=PIXEL_KERNEL, mask=keep & (magnitude >= 50))



normalization_mask = np.logical_or.reduce(list(masks.values()))
test_normalized = normalize(test_gray)
test_edges = detect_edges(test_normalized)

stage_names = ("Mask baseline", "Mask selisih setelah normalisasi", "Tepi baru Canny",
               "Mask gabungan OR", "Mask akhir morfologi")
stage_images = {name: np.zeros(test_gray.shape, dtype=bool) for name in stage_names}
rows = []
overlay = test_rgb.copy()
draw = ImageDraw.Draw(overlay)

for slot, ref_gray in references.items():
    mask = masks[slot]
    baseline = np.abs(test_gray - ref_gray) > DIFF_THRESHOLD
    ref_norm = normalize(ref_gray)
    changed = np.abs(test_normalized - ref_norm) > DIFF_THRESHOLD
    near_reference_edges = ndi.binary_dilation(detect_edges(ref_norm), structure=PIXEL_KERNEL)
    new_edges = ndi.binary_dilation(test_edges & ~near_reference_edges, structure=PIXEL_KERNEL)
    combined = changed | new_edges
    final = ndi.binary_closing(ndi.binary_opening(combined, structure=PIXEL_KERNEL),
                               structure=PIXEL_KERNEL)

    for stage, result in zip(stage_names, (baseline, changed, new_edges, combined, final)):
        stage_images[stage] |= result & mask

    baseline_ratio, final_ratio = baseline[mask].mean(), final[mask].mean()
    baseline_status = "Terisi mobil" if baseline_ratio >= OCCUPANCY_THRESHOLD else "Kosong"
    final_status = "Terisi mobil" if final_ratio >= OCCUPANCY_THRESHOLD else "Kosong"
    annotation = test_slots.get(slot)
    xml_status = annotation.get("occupied") if annotation is not None else None
    ground_truth = {"0": "Kosong", "1": "Terisi"}.get(xml_status, "Tidak tersedia")
    rows.append((slot, ground_truth, round(baseline_ratio * 100, 2), baseline_status,
                 round(final_ratio * 100, 2), final_status))
    color = "#ff5050" if final_status == "Terisi mobil" else "#55f294"
    draw.polygon(polygons[slot], outline=color, width=4)
    draw.text(polygons[slot][0], f"{slot}: {final_status}", fill=color,
              stroke_width=1, stroke_fill="black")

print("Sumber:", SOURCE_LABEL[MODE])
print("Petak | XML okupansi | Baseline (%) | Prediksi baseline | Lengkap (%) | Prediksi lengkap")
for row in rows:
    print(" | ".join(map(str, row)))
print("Jumlah petak kosong (metode lengkap):", sum(row[-1] == "Kosong" for row in rows))

Sumber: DEMO SINTETIS
Petak | XML okupansi | Baseline (%) | Prediksi baseline | Lengkap (%) | Prediksi lengkap
1 | Terisi | 68.99 | Terisi mobil | 97.23 | Terisi mobil
2 | Kosong | 6.62 | Kosong | 94.72 | Terisi mobil
3 | Kosong | 24.74 | Terisi mobil | 97.32 | Terisi mobil
4 | Terisi | 33.08 | Terisi mobil | 93.98 | Terisi mobil
Jumlah petak kosong (metode lengkap): 0


## 3. Simpan bukti progres

Hasil disimpan sebagai ringkasan visual PNG dan tabel CSV.

In [5]:
result_dir = Path("fiveclock_implementasi") / MODE
result_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 4, figsize=(14, 5.7), layout="constrained")
panels = [("Citra uji", test_rgb), ("Normalisasi pencahayaan", test_normalized),
          *stage_images.items(), ("Status petak", overlay)]
for ax, (label, shown) in zip(axes.flat, panels):
    ax.imshow(shown, cmap="gray")
    ax.set_title(label, fontsize=10)
    ax.axis("off")
fig.suptitle(f"Fiveclock · Implementasi Minggu 12 | {SOURCE_LABEL[MODE]}")
fig.savefig(result_dir / "bukti_progres.png", dpi=150)
with (result_dir / "status_petak.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["sumber", "petak", "okupansi_xml", "rasio_baseline_persen",
                     "prediksi_baseline", "rasio_lengkap_persen", "prediksi_lengkap"])
    writer.writerows((SOURCE_LABEL[MODE], *row) for row in rows)
print("Hasil tersimpan di", result_dir)
plt.show()

Hasil tersimpan di fiveclock_implementasi/demo
